# Terraria Chat — final project fixed

Ta wersja zostawia działające ładowanie baz bez zmian.

Dodane/poprawione:
- poprawniejsze parsowanie składników z separatorami `^`, np. `Blade of Grass1^Muramasa1`,
- pytania o `drop`, `loot`, `crate`, `chest`, `get from`, `open` idą do wiki zamiast do `item_stats`,
- pytania typu `craft + what do they do` pobierają jednocześnie recepturę i statystyki/tooltip,
- wiki używa dokładnego `title/page`, fuzzy title search i dopiero potem vector search,
- structured dane są zamieniane na fakty przed wysłaniem do LLM.


In [25]:
# # Instalacja potrzebnych bibliotek
# %pip install langchain langchain-core langchain-community langchain-ollama chromadb rapidfuzz


In [26]:
# Pobranie modeli Ollama
import os

for command in [
    "ollama pull nomic-embed-text",
    # "ollama pull qwen2.5:14b"
    # "ollama pull qwen3:8b"
    "ollama pull qwen2.5:14b"
]:
    print("Uruchamiam:", command)
    print("Kod wyjścia:", os.system(command))


Uruchamiam: ollama pull nomic-embed-text
Kod wyjścia: 0
Uruchamiam: ollama pull qwen2.5:14b
Kod wyjścia: 0


In [27]:
import json
import re
import unicodedata
from pathlib import Path

from rapidfuzz import process, fuzz

from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_core.prompts import ChatPromptTemplate


In [28]:
# Foldery z gotowymi bazami
WIKI_DB_DIR = Path("chroma_wiki_db")
STRUCTURED_DB_DIR = Path("chroma_structured_db")

# Lista nazw itemów
ITEM_NAMES_FILE = Path("item_names.json")


In [29]:
# Ten sam model embeddingów, którym była budowana baza
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

# LLM używany tylko do pytań opisowych z wiki
llm = OllamaLLM(
    # model="qwen2.5:14b",
    # model="qwen3:8b",
    model="qwen2.5:14b",
    temperature=0
)


In [30]:
# Ładowanie bazy wiki
wiki_db = Chroma(
    persist_directory=str(WIKI_DB_DIR),
    embedding_function=embeddings
)

# Ładowanie bazy structured
structured_db = Chroma(
    persist_directory=str(STRUCTURED_DB_DIR),
    embedding_function=embeddings
)

# Wczytanie nazw itemów do fuzzy matchingu
with ITEM_NAMES_FILE.open("r", encoding="utf-8") as file:
    item_names = json.load(file)

print("Wiki DB exists:", WIKI_DB_DIR.exists())
print("Structured DB exists:", STRUCTURED_DB_DIR.exists())
print("Item names:", len(item_names))
print("Wiki count:", wiki_db._collection.count())
print("Structured count:", structured_db._collection.count())


Wiki DB exists: True
Structured DB exists: True
Item names: 6352
Wiki count: 262167
Structured count: 10532


In [31]:
def normalize_text(text: str):
    # Usuwa polskie znaki i zamienia tekst na małe litery
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    return text


def normalize_for_matching(text: str):
    # Czyści tekst do porównywania nazw i wykrywania typów pytań
    text = normalize_text(text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [32]:
# Słowa, które nie są nazwą itemu i przeszkadzają w fuzzy matchingu
STOPWORDS = {
    "how", "to", "the", "a", "an", "is", "are", "what", "which", "with",
    "craft", "crafted", "crafting", "recipe", "use", "used", "for",
    "stats", "stat", "damage", "defense", "effect", "works",

    "jaki", "jakie", "jaka", "jak", "sie", "się", "ma", "maja", "mają",
    "co", "to", "czym", "jest", "dziala", "dzialaja", "dzialanie",
    "statystyki", "statystyk", "crafting", "receptura", "skladniki",
    "skladnikow", "uzyc", "uzywa", "uzycie", "dzialaja", "dzialają"
}


def strip_question_words(question: str):
    # Zostawia głównie fragment, który może być nazwą itemu
    normalized = normalize_for_matching(question)

    words = [
        word for word in normalized.split()
        if word not in STOPWORDS and len(word) > 1
    ]

    return " ".join(words)


In [33]:
# Normalizowane nazwy itemów
normalized_item_names = {
    normalize_for_matching(name): name
    for name in item_names
    if normalize_for_matching(name)
}

normalized_item_name_keys = list(normalized_item_names.keys())

# Aliasy bez spacji, np. "terraspark boots" -> "terrasparkboots"
item_aliases = {}

for normalized_name, original_name in normalized_item_names.items():
    item_aliases[normalized_name] = original_name
    item_aliases[normalized_name.replace(" ", "")] = original_name

item_alias_keys = list(item_aliases.keys())

print("Normalized item names:", len(normalized_item_name_keys))
print("Item aliases:", len(item_alias_keys))


Normalized item names: 6352
Item aliases: 12275


In [34]:
# Indeks tytułów stron wiki do dokładniejszego wyszukiwania
# Czytamy metadane partiami, bo cała wiki naraz może wywalić SQLite/Chroma.
wiki_title_lookup = {}

total = wiki_db._collection.count()
batch_size = 5000

for offset in range(0, total, batch_size):
    print(f"Ładuję metadata wiki {offset}-{min(offset + batch_size, total)} / {total}")

    wiki_meta_data = wiki_db._collection.get(
        include=["metadatas"],
        limit=batch_size,
        offset=offset
    )

    for metadata in wiki_meta_data["metadatas"]:
        for key in ["title", "page"]:
            title = metadata.get(key, "")

            if not title:
                continue

            normalized_title = normalize_for_matching(title)

            # Pomijamy bardzo krótkie tytuły, bo dają dużo fałszywych trafień
            if len(normalized_title) < 4:
                continue

            if normalized_title not in wiki_title_lookup:
                wiki_title_lookup[normalized_title] = title

wiki_title_keys = list(wiki_title_lookup.keys())

print("Wiki titles:", len(wiki_title_keys))


Ładuję metadata wiki 0-5000 / 262167
Ładuję metadata wiki 5000-10000 / 262167
Ładuję metadata wiki 10000-15000 / 262167
Ładuję metadata wiki 15000-20000 / 262167
Ładuję metadata wiki 20000-25000 / 262167
Ładuję metadata wiki 25000-30000 / 262167
Ładuję metadata wiki 30000-35000 / 262167
Ładuję metadata wiki 35000-40000 / 262167
Ładuję metadata wiki 40000-45000 / 262167
Ładuję metadata wiki 45000-50000 / 262167
Ładuję metadata wiki 50000-55000 / 262167
Ładuję metadata wiki 55000-60000 / 262167
Ładuję metadata wiki 60000-65000 / 262167
Ładuję metadata wiki 65000-70000 / 262167
Ładuję metadata wiki 70000-75000 / 262167
Ładuję metadata wiki 75000-80000 / 262167
Ładuję metadata wiki 80000-85000 / 262167
Ładuję metadata wiki 85000-90000 / 262167
Ładuję metadata wiki 90000-95000 / 262167
Ładuję metadata wiki 95000-100000 / 262167
Ładuję metadata wiki 100000-105000 / 262167
Ładuję metadata wiki 105000-110000 / 262167
Ładuję metadata wiki 110000-115000 / 262167
Ładuję metadata wiki 115000-12000

In [35]:
QUESTION_TYPES = {
    "recipe": [
        "craft", "crafted", "crafting", "recipe",
        "ingredient", "ingredients", "material", "materials",
        "receptur", "skladnik", "skladniki",
        "wytworz", "stworz", "zrob", "robic",
        "jak zrobic", "jak stworzyc", "jak scraftowac",
        "ile potrzebuje", "ile potrzeba", "how many materials"
    ],
    "item_stats": [
        "damage", "defense", "mana", "crit", "critical",
        "knockback", "sell", "buy", "rarity", "rare",
        "stat", "stats", "speed", "use time", "value", "tooltip",
        "obrazen", "obron", "atak", "statyst", "wartosc", "sprzedaz",

        # pytania o działanie itemu
        "dzial", "dzialanie", "effect", "effects", "works", "work",
        "use", "used for", "what do", "what does", "do they do",
        "does it do", "what does it do", "what do they do",
        "ability", "abilities"
    ],
    "wiki": [
        "co to", "czym jest", "opisz", "opis",
        "tell me", "describe", "what is"
    ]
}


In [36]:
def detect_question_types(question: str):
    # Wykrywa, czy pytanie dotyczy receptury, statystyk czy zwykłego opisu
    q = normalize_for_matching(question)

    matched_types = []

    for question_type, keywords in QUESTION_TYPES.items():
        if any(keyword in q for keyword in keywords):
            matched_types.append(question_type)

    if not matched_types:
        matched_types.append("wiki")

    return matched_types


In [37]:
def find_best_item(question: str, threshold: int = 65, allow_fuzzy: bool = True):
    # Szuka oficjalnej nazwy itemu w pytaniu
    question_normalized = normalize_for_matching(question)

    # Najpierw dokładne wystąpienie pełnej nazwy itemu w pytaniu
    exact_candidates = [
        normalized_name
        for normalized_name in normalized_item_name_keys
        if normalized_name in question_normalized
    ]

    if exact_candidates:
        best_exact = max(exact_candidates, key=len)
        return normalized_item_names[best_exact]

    if not allow_fuzzy:
        return None

    # Potem fuzzy matching na fragmencie bez słów pytających
    item_query = strip_question_words(question)

    if not item_query:
        return None

    item_query_no_spaces = item_query.replace(" ", "")

    match_1 = process.extractOne(
        item_query,
        item_alias_keys,
        scorer=fuzz.WRatio
    )

    match_2 = process.extractOne(
        item_query_no_spaces,
        item_alias_keys,
        scorer=fuzz.WRatio
    )

    matches = [match for match in [match_1, match_2] if match is not None]

    if not matches:
        return None

    matched_alias, score, index = max(matches, key=lambda x: x[1])

    print("FUZZY ITEM:", item_aliases[matched_alias], "| score:", score, "| query:", item_query)

    if score < threshold:
        return None

    return item_aliases[matched_alias]


In [ ]:
def strip_wiki_question_words(question: str):
    # Usuwa słowa pytające, ale zostawia ważne słowa z nazw typu "of" w "Wall of Flesh"
    stopwords = {
        "what", "is", "are", "the", "a", "an",
        "tell", "me", "about", "describe", "please",
        "co", "to", "czym", "jest", "opisz", "opis"
    }

    normalized = normalize_for_matching(question)

    words = [
        word for word in normalized.split()
        if word not in stopwords and len(word) > 1
    ]

    return " ".join(words)


def is_loot_question(question: str):
    # Pytania o dropy, loot, crate'y i chesty lepiej kierować do wiki
    q = normalize_for_matching(question)

    loot_keywords = [
        "drop", "drops", "loot", "get from", "obtain from",
        "open", "opening", "crate", "crates",
        "chest", "chests", "contain", "contains",
        "content", "contents", "reward", "rewards"
    ]

    return any(keyword in q for keyword in loot_keywords)


def find_best_wiki_title(question: str, threshold: int = 92):
    # Szuka najlepszego tytułu strony wiki dla pytania
    q = normalize_for_matching(question)
    query = strip_wiki_question_words(question)

    if not query:
        query = q

    # Najpierw dokładne dopasowanie całego tytułu w pytaniu
    exact_matches = [
        title_key for title_key in wiki_title_keys
        if title_key in q and (len(title_key) >= 6 or len(title_key.split()) >= 2)
    ]

    if exact_matches:
        best_key = max(exact_matches, key=lambda key: (len(key.split()), len(key)))
        return wiki_title_lookup[best_key]

    # Potem fuzzy matching po tytułach stron, a nie po treści chunków
    candidate_titles = [
        title_key for title_key in wiki_title_keys
        if len(title_key) >= 5
    ]

    match = process.extractOne(
        query,
        candidate_titles,
        scorer=fuzz.WRatio
    )

    if match is None:
        return None

    matched_key, score, index = match

    print("FUZZY WIKI TITLE:", wiki_title_lookup[matched_key], "| score:", score, "| query:", query)

    if score < threshold:
        return None

    return wiki_title_lookup[matched_key]


def search_wiki_by_title(question: str, k: int = 3):
    # Pobiera chunki konkretnej strony wiki po metadata title/page
    title = find_best_wiki_title(question)

    if not title:
        return []

    docs = []

    for metadata_field in ["title", "page"]:
        data = wiki_db._collection.get(
            where={metadata_field: {"$eq": title}},
            include=["metadatas", "documents"]
        )

        for metadata, document in zip(data["metadatas"], data["documents"]):
            docs.append(Document(
                page_content=document,
                metadata=metadata
            ))

        if docs:
            break

    def chunk_number(doc):
        try:
            return int(doc.metadata.get("chunk", 999999))
        except Exception:
            return 999999

    # Początkowe chunki zwykle mają definicję/opis strony.
    # Nie bierzemy za dużo, żeby model mniej mieszał sekcje.
    docs = sorted(docs, key=chunk_number)

    return docs[:k]


def search_wiki(question: str, k: int = 5):
    title_docs = search_wiki_by_title(question, k=6)

    if title_docs:
        return title_docs

    wiki_query = f"{question} Terraria wiki article overview main page"
    return wiki_db.similarity_search(wiki_query, k=k)


In [39]:
def get_structured_docs(question_type: str, item_name: str | None = None, k: int = 10):
    # Pobiera dane structured po metadanych, bez similarity search
    data = structured_db._collection.get(
        where={"type": {"$eq": question_type}},
        include=["metadatas", "documents"]
    )

    docs = []

    for metadata, document in zip(data["metadatas"], data["documents"]):
        if item_name and metadata.get("item_name") != item_name:
            continue

        docs.append(Document(
            page_content=document,
            metadata=metadata
        ))

    # Przy recepturach preferujemy aktualne receptury
    if question_type == "recipe":
        current_docs = [
            doc for doc in docs
            if str(doc.metadata.get("legacy", "")) == "0"
        ]

        if current_docs:
            docs = current_docs

    return docs[:k]


In [40]:
def deduplicate_docs(docs):
    # Usuwa powtarzające się dokumenty
    seen = set()
    unique_docs = []

    for doc in docs:
        key = (
            doc.metadata.get("source", ""),
            doc.metadata.get("type", ""),
            doc.metadata.get("item_name", ""),
            doc.page_content[:200]
        )

        if key in seen:
            continue

        seen.add(key)
        unique_docs.append(doc)

    return unique_docs


In [41]:
def retrieve_context(question: str, k: int = 10):
    # Najpierw próbuje użyć structured, a wiki dopiero jako fallback
    detected_types = detect_question_types(question)
    loot_question = is_loot_question(question)

    structured_types = [
        question_type for question_type in detected_types
        if question_type in ["recipe", "item_stats"]
    ]

    # Pytania o loot/drop/crate/chest kierujemy do wiki,
    # bo item_stats zwykle nie zawiera listy dropów.
    if loot_question:
        structured_types = []

    if structured_types:
        item_name = find_best_item(question, allow_fuzzy=True)
    else:
        item_name = find_best_item(question, allow_fuzzy=False)

    docs = []

    # Jeśli znaleziono item, ale pytanie było opisowe, bierzemy tooltip/statystyki.
    # Nie robimy tego dla pytań lootowych.
    if item_name and not structured_types and not loot_question:
        structured_types = ["item_stats"]

    # Structured pobieramy dokładnie po metadanych
    if structured_types:
        for question_type in structured_types:
            docs += get_structured_docs(
                question_type=question_type,
                item_name=item_name,
                k=k
            )

    # Wiki tylko jeśli structured nic nie znalazło
    if not docs:
        docs += search_wiki(question, k=5)

    docs = deduplicate_docs(docs)

    used_types = structured_types if docs and structured_types else ["wiki"]

    return docs, item_name, used_types


In [42]:
def clean_structured_text(text: str):
    # Poprawia czytelność, jeśli dane w bazie są sklejone bez nowych linii
    replacements = [
        ("Result amount:", "\nResult amount:"),
        ("Ingredients:", "\nIngredients:"),
        ("Crafting station:", "\nCrafting station:"),
        ("Legacy recipe:", "\nLegacy recipe:"),
        ("Version:", "\nVersion:"),

        ("name:", "\nname:"),
        ("itemid:", "\nitemid:"),
        ("internalname:", "\ninternalname:"),
        ("type:", "\ntype:"),
        ("tag:", "\ntag:"),
        ("rare:", "\nrare:"),
        ("sell:", "\nsell:"),
        ("tooltip:", "\ntooltip:"),
        ("damage:", "\ndamage:"),
        ("damagetype:", "\ndamagetype:"),
        ("defense:", "\ndefense:"),
        ("knockback:", "\nknockback:"),
        ("critical:", "\ncritical:"),
        ("usetime:", "\nusetime:"),
        ("mana:", "\nmana:"),
        ("velocity:", "\nvelocity:"),
        ("consumable:", "\nconsumable:"),
        ("placeable:", "\nplaceable:"),
    ]

    for old, new in replacements:
        text = text.replace(old, new)

    lines = [
        line for line in text.splitlines()
        if line.strip() and not line.strip().endswith(":")
    ]

    return "\n".join(lines).strip()


def detect_amount(question: str):
    # Wykrywa liczbę sztuk z pytania, np. "craft 10" albo "10 swords"
    match = re.search(r"\b\d+\b", question)

    if match:
        return int(match.group())

    return 1


def parse_number(value, default=1):
    # Wyciąga pierwszą liczbę z tekstu, np. "Result amount: 5"
    match = re.search(r"\d+", str(value))

    if match:
        return int(match.group())

    return default


def parse_single_ingredient(raw: str):
    # Rozpoznaje składnik i ilość, np. "Iron Bar8", "8x Iron Bar", "8 Iron Bar", "Iron Bar x8"
    raw = str(raw).strip()

    if not raw:
        return None

    # np. "1x Blade of Grass1" -> 1x Blade of Grass
    match = re.match(r"^(\d+)\s*x?\s+(.+)$", raw, flags=re.IGNORECASE)
    if match:
        amount = int(match.group(1))
        name = match.group(2).strip()

        trailing = re.match(r"^(.+?)(\d+)$", name)
        if trailing and int(trailing.group(2)) == amount:
            name = trailing.group(1).strip()

        return name, amount

    # np. "Iron Bar x8"
    match = re.match(r"^(.+?)\s*x\s*(\d+)$", raw, flags=re.IGNORECASE)
    if match:
        name = match.group(1).strip()
        amount = int(match.group(2))
        return name, amount

    # np. "Iron Bar8"
    match = re.match(r"^(.+?)(\d+)$", raw)
    if match:
        name = match.group(1).strip()
        amount = int(match.group(2))
        return name, amount

    return raw, 1


def parse_ingredients(line: str):
    # Zamienia linię Ingredients na listę par: (nazwa składnika, ilość)
    ingredients_text = line.replace("Ingredients:", "").strip()

    if not ingredients_text:
        return []

    # Ważne: dane często mają separator ^, np. "Iron Bar8^Wood2"
    parts = [
        ingredient.strip()
        for ingredient in re.split(r"\s*\^\s*|\s*,\s*|\s*;\s*", ingredients_text)
        if ingredient.strip()
    ]

    parsed = []

    for part in parts:
        ingredient = parse_single_ingredient(part)

        if ingredient:
            parsed.append(ingredient)

    return parsed


def structured_docs_to_facts(docs, question: str):
    # Zamienia structured docs na prosty format faktów dla LLM
    # Kod liczy ilości, LLM tylko ubiera odpowiedź w naturalny język.
    requested_amount = detect_amount(question)
    facts = [f"Requested final item amount: {requested_amount}"]

    for option_index, doc in enumerate(docs, start=1):
        doc_type = doc.metadata.get("type", "")
        item_name = doc.metadata.get("item_name", "")
        legacy = doc.metadata.get("legacy", "")
        text = clean_structured_text(doc.page_content)

        lines = [line.strip() for line in text.splitlines() if line.strip()]

        result_amount = 1
        for line in lines:
            if line.startswith("Result amount:"):
                result_amount = parse_number(line, default=1)

        crafts_needed = (requested_amount + result_amount - 1) // result_amount

        facts.append("")
        facts.append(f"Item: {item_name}")
        facts.append(f"Data type: {doc_type}")

        if doc_type == "recipe":
            facts.append(f"Recipe option: {option_index}")
            facts.append(f"Result amount per craft: {result_amount}")
            facts.append(f"Crafts needed for requested amount: {crafts_needed}")

        if legacy != "":
            facts.append(f"Legacy: {legacy}")

        for line in lines:
            if line.startswith("Recipe for:"):
                facts.append(line)

            elif line.startswith("Ingredients:"):
                ingredients = parse_ingredients(line)

                facts.append("Ingredients per craft:")

                for ingredient_name, ingredient_amount in ingredients:
                    facts.append(f"- {ingredient_amount}x {ingredient_name}")

                facts.append(f"Total ingredients for {requested_amount} requested item(s):")

                for ingredient_name, ingredient_amount in ingredients:
                    total_amount = ingredient_amount * crafts_needed
                    facts.append(f"- {total_amount}x {ingredient_name}")

            elif line.startswith("Crafting station:"):
                station = line.replace("Crafting station:", "").strip()
                facts.append(f"Crafting station: {station}")

            elif line.startswith("tooltip:"):
                tooltip = line.replace("tooltip:", "").strip()
                facts.append(f"Effects: {tooltip}")

            elif line.startswith("sell:"):
                sell = line.replace("sell:", "").strip()
                facts.append(f"Sell value: {sell}")

            elif line.startswith("rare:"):
                rare = line.replace("rare:", "").strip()
                facts.append(f"Rarity: {rare}")

            elif line.startswith("damage:"):
                damage = line.replace("damage:", "").strip()
                facts.append(f"Damage: {damage}")

            elif line.startswith("damagetype:"):
                damage_type = line.replace("damagetype:", "").strip()
                facts.append(f"Damage type: {damage_type}")

            elif line.startswith("defense:"):
                defense = line.replace("defense:", "").strip()
                facts.append(f"Defense: {defense}")

            elif line.startswith("knockback:"):
                knockback = line.replace("knockback:", "").strip()
                facts.append(f"Knockback: {knockback}")

            elif line.startswith("critical:"):
                critical = line.replace("critical:", "").strip()
                facts.append(f"Critical chance: {critical}")

            elif line.startswith("velocity:"):
                velocity = line.replace("velocity:", "").strip()
                facts.append(f"Velocity: {velocity}")

            elif line.startswith("type:"):
                item_type = line.replace("type:", "").strip()
                facts.append(f"Type: {item_type}")

            elif line.startswith("tag:"):
                tag = line.replace("tag:", "").strip()
                facts.append(f"Tag: {tag}")

    return "\n".join(facts).strip()


def answer_from_structured(docs):
    # Awaryjnie zwraca dane structured dokładnie z bazy
    if not docs:
        return "I cannot answer this based on the provided text."

    parts = []

    for doc in docs:
        doc_type = doc.metadata.get("type", "")
        item_name = doc.metadata.get("item_name", "")
        legacy = doc.metadata.get("legacy", "")

        header = f"=== {doc_type}: {item_name}"
        if legacy != "":
            header += f" | legacy: {legacy}"
        header += " ==="

        parts.append(header)
        parts.append(clean_structured_text(doc.page_content))

    return "\n\n".join(parts)


In [ ]:
system_prompt = """
You are a Terraria QA assistant.

Answer in the same language as the user's question.

Answer ONLY using the provided CONTEXT.
Do not use outside knowledge.
Do not guess.
Do not infer.
Do not add facts that are not explicitly present in CONTEXT.
Do not invent items, recipes, stats, numbers, NPCs, bosses, drops, strategies, or mechanics.

Do not list drops, loot, detailed statistics, version-specific notes, or long item lists unless the user explicitly asks about drops, loot, stats, or item lists.

If the user asks for a general description, give a short general description based only on CONTEXT.

If the answer is not directly stated in the CONTEXT, reply exactly:
"I cannot answer this based on the provided text."

Use a short answer.
For every factual claim, it must be directly supported by CONTEXT.

CONTEXT:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])

chain = prompt | llm


In [ ]:
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a Terraria assistant.

Answer in the same language as the user's question.

Use ONLY the FACTS below.
Do not use outside knowledge.
Do not invent, omit, or change item names, ingredients, stats, numbers, crafting stations, or effects.
Do not add commentary, opinions, platform/version information, edition information, or extra explanations unless they are explicitly present in FACTS.

Never change numbers from FACTS.
If the FACTS contain "Total ingredients", use those exact totals.
If an ingredient amount is listed as 8x, keep it as 8x.
Do not assume missing ingredients or deeper crafting trees.

For item stats, answer only with the stats present in FACTS.
For recipes, answer only with ingredients and crafting station.
Do not explain where an item comes from unless FACTS explicitly say it.

Do not mention legacy unless the user asks about legacy.
If there are alternative recipes, mention them as alternatives.

Give a natural but concise answer.
Avoid marketing-like phrases such as "unique", "impressive", "powerful", or "special" unless those exact words are in FACTS.

FACTS:
{context}
"""),
    ("human", "{question}")
])

structured_chain = structured_prompt | llm

In [45]:
def ask_terraria(question: str, show_context: bool = True):
    # Pobiera dokumenty i odpowiada z bazy albo przez LLM dla wiki
    docs, item_name, used_types = retrieve_context(question)

    if not docs:
        return "I cannot answer this based on the provided text."

    is_structured = any(
        question_type in used_types
        for question_type in ["recipe", "item_stats"]
    )

    if show_context:
        print("QUESTION TYPES:", used_types)
        print("MATCHED ITEM:", item_name)
        print("\nFOUND CONTEXT:")

        for i, doc in enumerate(docs, start=1):
            print(f"\n--- DOC {i} ---")
            print("metadata:", doc.metadata)

            if is_structured:
                print(clean_structured_text(doc.page_content[:1000]))
            else:
                print(doc.page_content[:1000])

        print("\nANSWER:")

    # Structured: fakty bierze kod, a LLM tylko pisze naturalną odpowiedź
    if is_structured:
        facts = structured_docs_to_facts(docs, question)

        return structured_chain.invoke({
            "question": question,
            "context": facts
        })

    # Wiki: zostaje stary prompt bez zmian
    context = "\n\n".join(
        f"SOURCE: {doc.metadata}\n{doc.page_content}"
        for doc in docs
    )

    answer = chain.invoke({
        "question": question,
        "context": context
    })

    return answer


## Testy

In [46]:
questions = [
    # 2 pytania o crafting
    "How do I craft night's Edge?",
    "How do I craft terraspark Boots?",

    # 2 pytania o statystyki
    "What stats does muramasa have?",
    "What stats does starfury have?",

    # 1 pytanie o crafting i statystyki naraz
    "How do I craft Lava Waders and what do they do?",

    # 2 pytania o crafting większej liczby przedmiotów
    "How many materials do I need to craft 10 terraspark boots?",
    "How many materials do I need to craft 5 night's edge?",

    # 3 pytania o bossy
    "What is queen bee?",
    "What is wall of flesh?",
    "Describe eye of cthulhu.",

    # drzewa
    "Tell me about trees",

    # skrzynki / crate'y / chesty / dropy
    "What are crates?",
    "What can I get from wooden crates?",
    "Tell me about gold chests.",
    "What can drop from sky crates?"
]

for question in questions:
    print("\n" + "=" * 80)
    print("QUESTION:", question)
    answer = ask_terraria(question, show_context=True)
    print("ANSWER:")
    print(answer)



QUESTION: How do I craft night's Edge?
QUESTION TYPES: ['recipe']
MATCHED ITEM: Night's Edge

FOUND CONTEXT:

--- DOC 1 ---
metadata: {'legacy': '0', 'item_name': "Night's Edge", 'type': 'recipe', 'source': 'structured', 'page': 'Recipes/Altar/register', 'resultid': '273'}
Recipe for: Night's Edge
Result amount: 1
Ingredients: 1x Blade of Grass1^Blood Butcherer1^Muramasa1^Volcano
Crafting station: Demon Altar
Legacy recipe: 0

--- DOC 2 ---
metadata: {'resultid': '273', 'source': 'structured', 'legacy': '0', 'type': 'recipe', 'page': 'Recipes/Altar/register', 'item_name': "Night's Edge"}
Recipe for: Night's Edge
Result amount: 1
Ingredients: 1x Blade of Grass1^Light's Bane1^Muramasa1^Volcano
Crafting station: Demon Altar
Legacy recipe: 0

ANSWER:
ANSWER:
To craft a Night's Edge, you have two options:

**Option 1:**
- Ingredients: 1 Blade of Grass, 1 Blood Butcherer, 1 Muramasa, 1 Volcano
- Crafting Station: Demon Altar

**Option 2:**
- Ingredients: 1 Blade of Grass, 1 Light's Bane, 1 

## Debug

In [47]:
for question in [
    "How to craft Night's Edge?",
    "jaki crafting i jakie statystyki ma terraspark buts?",
    "how to use Terraspark Boots?",
    "What is a Bee Queen?"
]:
    print("\nQUESTION:", question)
    print("normalized:", normalize_for_matching(question))
    print("clean item query:", strip_question_words(question))
    print("types:", detect_question_types(question))
    print("exact item:", find_best_item(question, allow_fuzzy=False))
    print("fuzzy item:", find_best_item(question, allow_fuzzy=True))



QUESTION: How to craft Night's Edge?
normalized: how to craft night s edge
clean item query: night edge
types: ['recipe']
exact item: Night's Edge
fuzzy item: Night's Edge

QUESTION: jaki crafting i jakie statystyki ma terraspark buts?
normalized: jaki crafting i jakie statystyki ma terraspark buts
clean item query: terraspark buts
types: ['recipe', 'item_stats']
exact item: None
FUZZY ITEM: Terraspark Boots | score: 90.32258064516128 | query: terraspark buts
fuzzy item: Terraspark Boots

QUESTION: how to use Terraspark Boots?
normalized: how to use terraspark boots
clean item query: terraspark boots
types: ['item_stats']
exact item: Terraspark Boots
fuzzy item: Terraspark Boots

QUESTION: What is a Bee Queen?
normalized: what is a bee queen
clean item query: bee queen
types: ['wiki']
exact item: None
FUZZY ITEM: Alien Queen Banner | score: 85.5 | query: bee queen
fuzzy item: Alien Queen Banner


## Debug wiki search


In [48]:
for question in [
    "What is Queen Bee?",
    "What is Wall of Flesh?",
    "Describe Eye of Cthulhu.",
    "Tell me about trees in Terraria.",
    "What is the melee class in Terraria?"
]:
    print("\nQUESTION:", question)
    print("wiki query:", strip_wiki_question_words(question))
    print("best wiki title:", find_best_wiki_title(question))
    docs = search_wiki_by_title(question, k=3)
    print("title docs:", len(docs))
    for doc in docs[:2]:
        print(doc.metadata)
        print(doc.page_content[:300])
        print()



QUESTION: What is Queen Bee?
wiki query: queen bee
best wiki title: Queen Bee
title docs: 3
{'chunk': 0, 'page': 'Queen Bee', 'title': 'Queen Bee', 'source': 'wiki', 'url': ''}
Page: Queen BeeThis is the main page whose information applies to the Desktop , Console , and Mobile versions of Terraria . For the differences of this information on Old-gen console and 3DS , see Legacy:Queen Bee . Queen Bee Map icon Classic Expert Master Statistics Type Boss Environment Bee Hive J

{'page': 'Queen Bee', 'title': 'Queen Bee', 'url': '', 'source': 'wiki', 'chunk': 1}
l always be dropped Bee Gun 33% Bee Keeper 33% The Bee's Knees (Desktop, Console and Mobile versions) 33% One of the following 4 items may be dropped Hive Wand 33% Bee Hat 11% Bee Shirt 11% Bee Pants 11% Honey Comb 33% Nectar 6.7% Queen of Bees (Desktop, Console and Mobile versions) 6.67% Honeyed Go


QUESTION: What is Wall of Flesh?
wiki query: wall of flesh
best wiki title: Wall of Flesh
title docs: 3
{'page': 'Wall of Flesh', 't